> **Executed on Databricks** (Free Edition, serverless compute, Unity Catalog).  
> Job run `814281740296490` · task `t00_setup_reset` · notebook `00_setup` · started 2026-10-11 12:54 UTC · result **SUCCESS**  
> Parameters: `catalog=workspace`, `reset=true`, `seed_demo_landing=false`

Saved with `scripts/export_databricks_run.py`: the cells and outputs below are exactly what the run produced.

# 00 · Setup
Creates the Unity Catalog schemas (`so_raw`, `so_bronze`, `so_silver`, `so_ops`), the landing volume and
every Delta table with its explicit schema, comments and primary key. Safe to run repeatedly.

| Widget | Meaning |
|---|---|
| `catalog` | Unity Catalog to use. Free Edition's default catalog is `workspace`. |
| `reset` | `true` drops and recreates every table. Use only to start over. |
| `seed_demo_landing` | `true` copies the demo batches in `data/demo_landing` from the repo into the volume. |

In [1]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("reset", "false", ["false", "true"])
dbutils.widgets.dropdown("seed_demo_landing", "true", ["true", "false"])

In [2]:
%run ./_bootstrap

repo root: /Workspace/Users/<your-email>/stackoverflow-lakehouse


In [3]:
import shutil

from so_lakehouse.ddl import setup

cfg = build_config()
setup(spark, cfg, reset=widget("reset") == "true")

dropped workspace.so_bronze.questions
created workspace.so_bronze.questions
dropped workspace.so_bronze.answers
created workspace.so_bronze.answers
dropped workspace.so_bronze.deleted_questions
created workspace.so_bronze.deleted_questions
dropped workspace.so_bronze.question_snapshots
created workspace.so_bronze.question_snapshots
dropped workspace.so_bronze.quarantine_records
created workspace.so_bronze.quarantine_records
dropped workspace.so_silver.questions
created workspace.so_silver.questions
dropped workspace.so_silver.question_tags
created workspace.so_silver.question_tags
dropped workspace.so_silver.answers
created workspace.so_silver.answers
dropped workspace.so_silver.users
created workspace.so_silver.users
dropped workspace.so_silver.question_snapshots
created workspace.so_silver.question_snapshots
dropped workspace.so_silver.post_text_masked
created workspace.so_silver.post_text_masked
dropped workspace.so_silver.dq_quarantine
created workspace.so_silver.dq_quarantine
drop

In [4]:
def copy_tree_plain(src, dst):
    """Copy files only (no permission/timestamp copying, which UC volumes do not support)."""
    copied = 0
    for folder, _dirs, files in os.walk(src):
        target = os.path.join(dst, os.path.relpath(folder, src))
        os.makedirs(target, exist_ok=True)
        for name in files:
            shutil.copyfile(os.path.join(folder, name), os.path.join(target, name))
            copied += 1
    return copied


if widget("seed_demo_landing") == "true":
    n = copy_tree_plain(os.path.join(REPO_ROOT, "data", "demo_landing"), cfg.landing_root)
    print(f"copied {n} files into {cfg.landing_root}")

for lt in ("full", "incremental"):
    folder = os.path.join(cfg.landing_root, lt)
    print(lt, sorted(os.listdir(folder)) if os.path.isdir(folder) else "(none)")

full ['full_2025-01-01_2025-04-01', 'full_2025-04-01_2025-07-01']
incremental ['incr_20260925T155859Z', 'incr_20261007T155914Z', 'incr_20261007T160033Z']


In [5]:
display(spark.sql(f"SHOW TABLES IN {cfg.schema_name('silver')}"))

database,tableName,isTemporary
so_silver,answers,False
so_silver,dq_quarantine,False
so_silver,post_text_masked,False
so_silver,question_snapshots,False
so_silver,question_tags,False
so_silver,questions,False
so_silver,users,False
